# Entrenar la voz propia de C&M Labs

Este cuaderno afina una voz base en español latino (`es_MX/ald/medium`) con tus grabaciones, y produce el archivo de voz `cm_labs.onnx` que usa el Agente Creador de Video.

**Antes de empezar:**

1. **Activa la tarjeta gráfica:** menú **Entorno de ejecución → Cambiar tipo de entorno de ejecución → Acelerador de hardware: GPU (T4)** → Guardar.
2. **Sube el conjunto de grabaciones a tu Google Drive:** crea la carpeta **`voz_propia_cm`** en «Mi unidad» y deja dentro el archivo **`dataset_voz_propia.zip`**.
3. Ejecuta las celdas **en orden**, una por una, con el botón ▶ de cada celda (o `Ctrl + Enter`).

**Si Colab te desconecta:** vuelve a ejecutar todas las celdas desde el principio. El avance del entrenamiento se guarda en tu Drive y continúa donde iba.

## 1. Comprobar la tarjeta gráfica

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv || echo "⚠ No hay tarjeta gráfica: activa la GPU en Entorno de ejecución → Cambiar tipo de entorno de ejecución."

## 2. Conectar tu Google Drive

Te pedirá permiso para acceder a tu Drive: acéptalo con la cuenta donde subiste el archivo.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

import os
CARPETA = "/content/drive/MyDrive/voz_propia_cm"   # tu carpeta en Drive
ZIP = f"{CARPETA}/dataset_voz_propia.zip"
os.makedirs(f"{CARPETA}/entrenamiento", exist_ok=True)
print("✓ Drive conectado" if os.path.exists(ZIP) else f"⚠ No encuentro {ZIP}. Revisa el nombre de la carpeta y del archivo.")

## 3. Descomprimir las grabaciones

In [ ]:
!rm -rf /content/dataset && mkdir -p /content/dataset && unzip -q "{ZIP}" -d /content/dataset
!echo "Grabaciones: $(ls /content/dataset/wavs | wc -l)" && head -3 /content/dataset/metadata.csv

## 4. Instalar Piper (herramienta de entrenamiento)

Tarda de 5 a 8 minutos y llena la pantalla de mensajes técnicos: es normal. Los avisos en amarillo («performance hint», «FutureWarning») no son errores. Al final debe decir **«✓ Piper listo para entrenar»**.

In [ ]:
%cd /content
!rm -rf piper1-gpl && git clone -q https://github.com/OHF-Voice/piper1-gpl.git
%cd /content/piper1-gpl
!python3 -m pip install -q -e ".[train]"
!bash ./build_monotonic_align.sh
# Componente de pronunciación (espeak): se construye con scikit-build, cmake y ninja (3 a 5 minutos).
!python3 -m pip install -q "scikit-build<1" "cmake>=3.18,<4" ninja
!python3 setup.py build_ext --inplace
!python3 -c "import piper.espeakbridge, piper.train; print('✓ Piper listo para entrenar')"

## 5. Descargar la voz base (solo la primera vez)

Unos 850 MB. Se guarda en tu Drive para no descargarla de nuevo si Colab te desconecta.

In [ ]:
BASE = f"{CARPETA}/base_es_MX_ald_medium.ckpt"
if not os.path.exists(BASE):
    !wget -q --show-progress -O "{BASE}" "https://huggingface.co/datasets/rhasspy/piper-checkpoints/resolve/main/es/es_MX/ald/medium/epoch%3D9999-step%3D1753600.ckpt"
print("✓ Voz base lista:", round(os.path.getsize(BASE) / 1e6), "MB")

## 6. Entrenar

- **La primera vez** parte de la voz base. **Si ya habías empezado**, continúa desde el último punto de guardado que quedó en tu Drive.
- Con unas 200 grabaciones, cada vuelta completa a los datos (una **época**) tarda unos segundos. El límite está en 1.500 épocas, pero **no hace falta llegar al final**: desde la época 300 a 500 ya suele sonar a ti. Ve a la celda 7 para escucharla y detén esta celda (botón ■) cuando te guste.
- Los puntos de guardado quedan en `voz_propia_cm/entrenamiento` de tu Drive.

In [ ]:
import glob
RAIZ = f"{CARPETA}/entrenamiento"
CONFIG = f"{CARPETA}/cm_labs.onnx.json"
ultimos = sorted(glob.glob(f"{RAIZ}/lightning_logs/version_*/checkpoints/last.ckpt"), key=os.path.getmtime)

if ultimos:
    print("↺ Continuando desde:", ultimos[-1])
    inicio = f'--ckpt_path "{ultimos[-1]}"'
else:
    print("▶ Primera vez: se parte de la voz base")
    inicio = f'--model.warmstart_ckpt "{BASE}"'

!cd /content/piper1-gpl && python3 -m piper.train fit \
  --data.voice_name "cm_labs" \
  --data.csv_path /content/dataset/metadata.csv \
  --data.audio_dir /content/dataset/wavs \
  --model.sample_rate 22050 \
  --data.espeak_voice "es-419" \
  --data.cache_dir /content/cache \
  --data.config_path "{CONFIG}" \
  --data.batch_size 16 \
  --data.num_workers 2 \
  --trainer.default_root_dir "{RAIZ}" \
  --trainer.check_val_every_n_epoch 25 \
  --trainer.max_epochs 1500 \
  {inicio}

## 7. Escuchar cómo va

Puedes ejecutar esta celda mientras entrenas: abre la celda, pulsa ▶ y espera. Convierte el último punto de guardado y lee unas frases que **no** estaban en tus grabaciones.

Si suena metálico o raro, deja entrenar más. Si ya suena como tú, detén la celda 6 y pasa a la 8.

In [ ]:
import glob, wave
from IPython.display import Audio, display

ultimo = sorted(glob.glob(f"{RAIZ}/lightning_logs/version_*/checkpoints/last.ckpt"), key=os.path.getmtime)[-1]
print("Punto de guardado:", ultimo)
!cd /content/piper1-gpl && python3 -m piper.train.export_onnx --checkpoint "{ultimo}" --output-file /content/prueba.onnx
!cp "{CONFIG}" /content/prueba.onnx.json

from piper import PiperVoice
voz = PiperVoice.load("/content/prueba.onnx")
PRUEBAS = [
    "Hola, soy la voz de C y M Labs. Hoy te voy a mostrar cómo analizar las ventas de tu negocio.",
    "Primera alerta: la sede del norte vende un veintiocho por ciento menos desde julio.",
    "¿Quieres que tu equipo trabaje así? Escríbenos y te contamos cómo.",
]
for i, texto in enumerate(PRUEBAS, 1):
    salida = f"/content/prueba_{i}.wav"
    with wave.open(salida, "wb") as w:
        voz.synthesize_wav(texto, w)
    print(texto); display(Audio(salida))

## 8. Guardar la voz final en tu Drive

Exporta el último punto de guardado como **`cm_labs.onnx`** (y su configuración **`cm_labs.onnx.json`**) en tu carpeta `voz_propia_cm`. Descárgalos y ponlos juntos en tu computador.

In [ ]:
ultimo = sorted(glob.glob(f"{RAIZ}/lightning_logs/version_*/checkpoints/last.ckpt"), key=os.path.getmtime)[-1]
!cd /content/piper1-gpl && python3 -m piper.train.export_onnx --checkpoint "{ultimo}" --output-file "{CARPETA}/cm_labs.onnx"
print("✓ Voz guardada en tu Drive: voz_propia_cm/cm_labs.onnx y cm_labs.onnx.json")
print("En el agente: config/canal.toml → [voz] modo = \"sintetica\" y modelo_piper = \"ruta/a/cm_labs.onnx\"")